# **Step 1: Protein Preparation**

In [ ]:
# =============================================================================
#  Protein Preparation Pipeline for Molecular Docking
# =============================================================================
#  Pipeline : PDBFixer -> BioPython -> PropKa -> pdb2pqr -> OpenMM EM
#  Output   : Protonated, energy-minimised APO structure ready for docking
#
#  Usage    : Run this cell in Google Colab. Adjust parameters below as needed.
# =============================================================================


# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

structure_source = "Upload PDB file"  # @param ["Upload PDB file", "Fetch from RCSB PDB"]
pdb_id = "3MXF"  # @param {type:"string"}
target_pH = 7.4  # @param {type:"number"}
restraint_k = 500  # @param {type:"number"}
convergence_tol = 10  # @param {type:"number"}
max_iterations = 2000  # @param {type:"integer"}


# ---------------------------------------------------------------------------
#  STEP 0 : Install Dependencies
# ---------------------------------------------------------------------------

import subprocess, sys

print("=" * 72)
print("  STEP 0 | Installing Dependencies")
print("=" * 72)

_packages = {
    "openmm":    "openmm",
    "pdbfixer":  "pdbfixer",
    "biopython": "Bio",
    "propka":    "propka",
    "pdb2pqr":   "pdb2pqr",
    "requests":  "requests",
    "numpy":     "numpy",
}

for pkg_name, import_name in _packages.items():
    try:
        __import__(import_name)
        print(f"  [OK]  {pkg_name:24s} already installed")
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pkg_name],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        print(f"  [OK]  {pkg_name:24s} installed")

print("  All dependencies ready.\n")


# ---------------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------------

import os, io, warnings, tempfile, shutil
import numpy as np
import requests

from pathlib import Path
from pdbfixer import PDBFixer
from openmm.app import (
    PDBFile, ForceField, Modeller, Simulation, PDBReporter,
)
from openmm import (
    LangevinMiddleIntegrator, CustomExternalForce, unit, Platform,
)
from Bio.PDB import PDBParser, PDBIO, Select

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------------
#  Helper Utilities
# ---------------------------------------------------------------------------

INTERMEDIATE_DIR = Path("/content/prep_intermediates")
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

def _save(structure_path: str, tag: str) -> Path:
    """Copy a file into the intermediate directory with a numbered tag."""
    dest = INTERMEDIATE_DIR / tag
    shutil.copy2(structure_path, dest)
    return dest

def _atom_count(pdb_path):
    """Return total atom count from a PDB file."""
    count = 0
    with open(pdb_path) as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                count += 1
    return count

def _print_header(step_num, title):
    print("\n" + "=" * 72)
    print(f"  STEP {step_num} | {title}")
    print("=" * 72)


# ---------------------------------------------------------------------------
#  STEP 1 : Protein Input
# ---------------------------------------------------------------------------

_print_header(1, "Protein Input")

if structure_source == "Upload PDB file":
    # Google Colab file upload widget
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded. Please re-run and select a PDB file.")
    upload_name = list(uploaded.keys())[0]
    raw_path = str(INTERMEDIATE_DIR / "01_raw_input.pdb")
    with open(raw_path, "wb") as f:
        f.write(uploaded[upload_name])
    print(f"  Uploaded '{upload_name}' -> {raw_path}")

elif structure_source == "Fetch from RCSB PDB":
    if not pdb_id.strip():
        raise ValueError("Please enter a PDB accession ID in the pdb_id parameter.")
    _id = pdb_id.strip().upper()
    url = f"https://files.rcsb.org/download/{_id}.pdb"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise RuntimeError(f"Could not fetch PDB {_id} (HTTP {resp.status_code}).")
    raw_path = str(INTERMEDIATE_DIR / "01_raw_input.pdb")
    with open(raw_path, "w") as f:
        f.write(resp.text)
    print(f"  Fetched {_id} from RCSB -> {raw_path}")

else:
    raise ValueError("Invalid structure_source selection.")


# ---------------------------------------------------------------------------
#  STEP 2 : Initial Structure Audit
# ---------------------------------------------------------------------------

_print_header(2, "Initial Structure Audit")

parser = PDBParser(QUIET=True)
structure = parser.get_structure("input", raw_path)

models      = list(structure.get_models())
chains      = [c.id for c in models[0].get_chains()]
residues    = [r for r in models[0].get_residues() if r.id[0] == " "]
hetatm_res  = [r for r in models[0].get_residues() if r.id[0] not in (" ", "W")]
waters      = [r for r in models[0].get_residues() if r.id[0] == "W"]
atoms       = list(models[0].get_atoms())
altloc_atoms = [a for a in atoms if a.get_altloc() not in (" ", "", "A")]
ins_code_res = [r for r in models[0].get_residues() if r.id[2].strip()]

print(f"  {'Models':<40s} {len(models)}")
print(f"  {'Chains':<40s} {chains}")
print(f"  {'Standard AA residues':<40s} {len(residues)}")
print(f"  {'HETATM residues':<40s} {len(hetatm_res)}")
print(f"  {'Water molecules':<40s} {len(waters)}")
print(f"  {'Total atoms':<40s} {len(atoms)}")
print(f"  {'Alt-conformation atoms':<40s} {len(altloc_atoms)}")
print(f"  {'Residues with insertion codes':<40s} {len(ins_code_res)}")

# Disulfide bond scan
print("\n  Scanning for disulfide bonds (CYS Sg distance < 2.5 A) ...")
cys_sg = [
    (r, a) for r in residues if r.get_resname() == "CYS"
    for a in r.get_atoms() if a.get_name() == "SG"
]
disulfides = []
for i in range(len(cys_sg)):
    for j in range(i + 1, len(cys_sg)):
        d = cys_sg[i][1] - cys_sg[j][1]
        if d < 2.5:
            disulfides.append((cys_sg[i][0], cys_sg[j][0], d))
if disulfides:
    for r1, r2, d in disulfides:
        print(f"    Disulfide: {r1.get_resname()} {r1.id[1]} -- "
              f"{r2.get_resname()} {r2.id[1]}  ({d:.2f} A)")
else:
    print("  No disulfide bonds detected.")

# B-factor distribution (Ca only)
print("\n  B-factor distribution (Ca only) ...")
ca_bfactors = np.array([
    a.get_bfactor() for r in residues for a in r.get_atoms()
    if a.get_name() == "CA"
])
if len(ca_bfactors) > 0:
    mean_b = ca_bfactors.mean()
    std_b  = ca_bfactors.std()
    print(f"    Mean   : {mean_b:.1f} A^2")
    print(f"    Median : {np.median(ca_bfactors):.1f} A^2")
    print(f"    Std    : {std_b:.1f} A^2")
    print(f"    Max    : {ca_bfactors.max():.1f} A^2")
    threshold = mean_b + 2 * std_b
    n_high = int(np.sum(ca_bfactors > threshold))
    if n_high > 0:
        print(f"    WARNING: {n_high} residue(s) above threshold ({threshold:.1f} A^2)")
    else:
        print("    No high B-factor residues detected.")


# ---------------------------------------------------------------------------
#  STEP 3 : Chain Selection
# ---------------------------------------------------------------------------

_print_header(3, "Chain Selection")

print(f"  Chains available: {chains}")

if len(chains) == 1:
    selected_chain = chains[0]
    print(f"  Single chain '{selected_chain}' -- no selection needed.")
else:
    # Keep all chains; user can modify this list to select specific chains
    selected_chain = chains
    print(f"  Multiple chains found. Keeping all: {selected_chain}")
    print("  (Edit 'selected_chain' in this block to restrict if needed.)")

# Write chain-selected structure
class ChainSelect(Select):
    def __init__(self, chain_ids):
        self.chain_ids = chain_ids if isinstance(chain_ids, list) else [chain_ids]
    def accept_chain(self, chain):
        return chain.id in self.chain_ids

chain_out = str(INTERMEDIATE_DIR / "02_chain_selected.pdb")
io_pdb = PDBIO()
io_pdb.set_structure(structure)
io_pdb.save(chain_out, ChainSelect(selected_chain))
print(f"  Chain-selected structure -> {chain_out}")


# ---------------------------------------------------------------------------
#  STEP 4 : PDBFixer -- Gaps, Missing Heavy Atoms, APO Conversion
# ---------------------------------------------------------------------------

_print_header(4, "PDBFixer: Gaps, Missing Heavy Atoms, APO Conversion")

fixer = PDBFixer(filename=chain_out)

# Missing residues in gaps
fixer.findMissingResidues()
n_gaps = sum(len(v) for v in fixer.missingResidues.values())
if n_gaps > 0:
    print(f"  Found {n_gaps} missing residue(s) in sequence gaps -- adding ...")
    fixer.findMissingResidues()
else:
    print("  No sequence gaps.")

# Non-standard residues
fixer.findNonstandardResidues()
if fixer.nonstandardResidues:
    print(f"  Replacing {len(fixer.nonstandardResidues)} non-standard residue(s) ...")
    fixer.replaceNonstandardResidues()
else:
    print("  No non-standard residues.")

# Remove all heteroatoms and water (APO conversion)
print("  Removing all heteroatoms and water (APO conversion) ...")
fixer.removeHeterogens(True)

# Missing heavy atoms
fixer.findMissingAtoms()
n_missing = sum(len(v) for v in fixer.missingAtoms.values())
n_terminal = sum(len(v) for v in fixer.missingTerminals.values())
print(f"  Missing heavy atoms: {n_missing}  |  Missing terminal atoms: {n_terminal}")
fixer.addMissingAtoms()
print("  All missing heavy atoms added.")

# Note: hydrogens deferred to pdb2pqr for pH-aware placement
print("  Hydrogens deferred to pdb2pqr (Step 7) for pH-aware placement.")

apo_path = str(INTERMEDIATE_DIR / "03_pdbfixer_apo.pdb")
PDBFile.writeFile(fixer.topology, fixer.positions, open(apo_path, "w"))
print(f"  PDBFixer output -> {apo_path}")


# ---------------------------------------------------------------------------
#  STEP 5 : BioPython -- Alt-Conformers, Insertion Codes, Single Model
# ---------------------------------------------------------------------------

_print_header(5, "BioPython: Alt-Conformers, Insertion Codes, Single Model")

structure = parser.get_structure("apo", apo_path)

# Normalise alt-conformer flags to blank and set occupancy to 1.0
n_altloc_fixed = 0
for atom in structure.get_atoms():
    if atom.get_altloc() not in (" ", ""):
        atom.set_altloc(" ")
        atom.set_occupancy(1.0)
        n_altloc_fixed += 1
print(f"  Normalised {n_altloc_fixed} altloc atoms -> blank (occupancy 1.0).")

# Renumber residues sequentially starting from 1 per chain.
# pdb2pqr's debumper interprets numbering gaps as structural gaps, so
# sequential numbering is required to prevent spurious "gap" errors.
n_renumbered = 0
for model in structure:
    for chain in model:
        new_id = 1
        for residue in list(chain.get_residues()):
            old_id = residue.id
            if old_id[1] != new_id or old_id[2] != " ":
                residue.id = (old_id[0], new_id, " ")
                n_renumbered += 1
            new_id += 1
if n_renumbered > 0:
    print(f"  Renumbered {n_renumbered} residue(s) to sequential IDs (required by pdb2pqr).")
else:
    print("  Residue numbering already sequential.")

cleaned_path = str(INTERMEDIATE_DIR / "04_biopy_cleaned.pdb")
io_pdb.set_structure(structure)
io_pdb.save(cleaned_path)
n_res = sum(1 for r in structure.get_residues() if r.id[0] == " ")
print(f"  Cleaned structure -> {cleaned_path}  ({n_res} residues)")


# ---------------------------------------------------------------------------
#  STEP 6 : PropKa -- Per-Residue pKa Prediction
# ---------------------------------------------------------------------------

_print_header(6, "PropKa: Per-Residue pKa Prediction")

propka_report_path = str(INTERMEDIATE_DIR / "05_propka_report.pka")

try:
    from propka.run import single as propka_single
    propka_mol = propka_single(cleaned_path, optargs=["--pH", str(target_pH)])

    # Write the PropKa report
    with open(propka_report_path, "w") as pka_out:
        propka_mol.write_pka(pka_out)
    print(f"  PropKa report -> {propka_report_path}")

    # Summarise titratable residues with shifted pKa
    print(f"\n  Titratable residues with pKa shifted by > 1 unit from default (pH {target_pH}):")
    _defaults = {"ASP": 3.8, "GLU": 4.5, "HIS": 6.5, "CYS": 9.0, "TYR": 10.5,
                 "LYS": 10.5, "ARG": 12.5}
    shifted = []
    for group in propka_mol.conformations["AVR"].groups:
        resname = group.residue_type
        if resname in _defaults:
            delta = abs(group.pka_value - _defaults[resname])
            if delta > 1.0:
                shifted.append((resname, group.atom.res_num, group.pka_value, delta))
    if shifted:
        for rn, rnum, pka, d in shifted:
            print(f"    {rn:>3s} {rnum:<5d}  pKa = {pka:5.2f}  (delta = {d:+.2f})")
    else:
        print("    None -- all titratable residues near default pKa values.")

except Exception as e:
    print(f"  WARNING: PropKa standalone run failed ({e}).")
    print("  pdb2pqr will run PropKa internally in Step 7.")


# ---------------------------------------------------------------------------
#  STEP 7 : pdb2pqr -- Protonation + H-Bond Network Optimisation
# ---------------------------------------------------------------------------

_print_header(7, "pdb2pqr: Protonation + H-Bond Network Optimisation")

print(f"  Running pdb2pqr (AMBER FF, PropKa, pH {target_pH}) ...")
print("    - Assigns HID / HIE / HIP per His residue")
print("    - Protonates Asp / Glu / Cys / Tyr based on predicted pKa")
print("    - Optimises Asn / Gln / His ring orientations")

pqr_out   = str(INTERMEDIATE_DIR / "06_protonated_hbond.pqr")
pdb_h_out = str(INTERMEDIATE_DIR / "06_protonated_hbond.pdb")

pdb2pqr_cmd = [
    sys.executable, "-m", "pdb2pqr",
    "--ff", "AMBER",
    "--ffout", "AMBER",
    "--with-ph", str(target_pH),
    "--titration-state-method", "propka",
    "--pdb-output", pdb_h_out,
    cleaned_path,
    pqr_out,
]

result = subprocess.run(pdb2pqr_cmd, capture_output=True, text=True)

# If pdb2pqr fails (commonly due to residual gap detection in the debumper),
# retry with --nodebump to skip steric clash resolution. The subsequent
# OpenMM energy minimisation (Step 8) will resolve any remaining clashes.
if result.returncode != 0:
    print("  WARNING: pdb2pqr debumper failed -- retrying with --nodebump ...")
    print("  (Steric clashes will be resolved during energy minimisation.)")
    pdb2pqr_cmd_nodebump = [
        sys.executable, "-m", "pdb2pqr",
        "--ff", "AMBER",
        "--ffout", "AMBER",
        "--with-ph", str(target_pH),
        "--titration-state-method", "propka",
        "--nodebump",
        "--pdb-output", pdb_h_out,
        cleaned_path,
        pqr_out,
    ]
    result = subprocess.run(pdb2pqr_cmd_nodebump, capture_output=True, text=True)
    if result.returncode != 0:
        print("  ERROR: pdb2pqr failed even with --nodebump. Stderr:")
        print(result.stderr)
        raise RuntimeError("pdb2pqr failed.")

print("  pdb2pqr completed successfully.")

# Count hydrogens added
n_H = 0
with open(pdb_h_out) as f:
    for line in f:
        if line.startswith("ATOM") and line[76:78].strip() == "H":
            n_H += 1
print(f"  Total hydrogens placed: {n_H}")

# Save unminimised protonated structure
unmin_path = str(INTERMEDIATE_DIR / "07_unminimised.pdb")
shutil.copy2(pdb_h_out, unmin_path)
print(f"  Unminimised protonated structure -> {unmin_path}")


# ---------------------------------------------------------------------------
#  STEP 8 : OpenMM -- Restrained Energy Minimisation (AMBER14 + GBN2)
# ---------------------------------------------------------------------------

_print_header(8, "OpenMM: Restrained Energy Minimisation (AMBER14 + GBN2)")

print(f"  Force field  : AMBER14-all + implicit/gbn2 (Generalised Born Neck 2)")
print(f"  Restraint    : k = {restraint_k} kJ/mol/nm^2 on all heavy atoms")
print(f"  Convergence  : {convergence_tol} kJ/mol/nm  |  max {max_iterations} iterations")

# --- FIX: Remove any residual HOH records before OpenMM loads the file ---
with open(pdb_h_out, 'r') as f:
    lines = f.readlines()
cleaned_lines = [l for l in lines if 'HOH' not in l and 'WAT' not in l]
n_removed = len(lines) - len(cleaned_lines)
if n_removed > 0:
    print(f"  Removed {n_removed} residual water line(s) before minimisation.")
with open(pdb_h_out, 'w') as f:
    f.writelines(cleaned_lines)
# -------------------------------------------------------------------------

# Load the protonated structure
pdb_in = PDBFile(pdb_h_out)

# ---------------------------------------------------------------------------
#  STEP 8 : OpenMM -- Restrained Energy Minimisation (AMBER14 + GBN2)
# ---------------------------------------------------------------------------

_print_header(8, "OpenMM: Restrained Energy Minimisation (AMBER14 + GBN2)")

print(f"  Force field  : AMBER14-all + implicit/gbn2 (Generalised Born Neck 2)")
print(f"  Restraint    : k = {restraint_k} kJ/mol/nm^2 on all heavy atoms")
print(f"  Convergence  : {convergence_tol} kJ/mol/nm  |  max {max_iterations} iterations")

# Load the protonated structure
pdb_in = PDBFile(pdb_h_out)

# Build the system with AMBER14 and GBN2 implicit solvent
ff = ForceField("amber14-all.xml", "implicit/gbn2.xml")
system = ff.createSystem(pdb_in.topology, nonbondedCutoff=1.0 * unit.nanometers)
print("  System built with GBN2 implicit solvent.")

# Add harmonic positional restraints on heavy atoms
restraint_force = CustomExternalForce(
    "0.5 * k * ((x - x0)^2 + (y - y0)^2 + (z - z0)^2)"
)
restraint_force.addGlobalParameter("k", restraint_k * unit.kilojoules_per_mole / unit.nanometers**2)
restraint_force.addPerParticleParameter("x0")
restraint_force.addPerParticleParameter("y0")
restraint_force.addPerParticleParameter("z0")

n_restrained = 0
for i, atom in enumerate(pdb_in.topology.atoms()):
    if atom.element.symbol != "H":
        pos = pdb_in.positions[i]
        restraint_force.addParticle(
            i, [pos.x, pos.y, pos.z]
        )
        n_restrained += 1

system.addForce(restraint_force)
print(f"  Harmonic restraints on {n_restrained} heavy atoms.")

# Set up the integrator and simulation (integrator unused; only minimisation)
integrator = LangevinMiddleIntegrator(
    300 * unit.kelvin, 1.0 / unit.picoseconds, 0.002 * unit.picoseconds
)
simulation = Simulation(pdb_in.topology, system, integrator)
simulation.context.setPositions(pdb_in.positions)

# Record energy before minimisation
state_before = simulation.context.getState(getEnergy=True, getPositions=True)
e_before = state_before.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
pos_before = state_before.getPositions(asNumpy=True).value_in_unit(unit.angstroms)

print("  Minimising ...")
import time
t0 = time.time()
simulation.minimizeEnergy(
    tolerance=convergence_tol * unit.kilojoules_per_mole / unit.nanometers,
    maxIterations=max_iterations,
)
wall_time = time.time() - t0

# Record energy and positions after minimisation
state_after = simulation.context.getState(getEnergy=True, getPositions=True)
e_after = state_after.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
pos_after = state_after.getPositions(asNumpy=True).value_in_unit(unit.angstroms)

# Compute heavy-atom RMSD
heavy_idx = [
    i for i, a in enumerate(pdb_in.topology.atoms()) if a.element.symbol != "H"
]
delta = pos_after[heavy_idx] - pos_before[heavy_idx]
rmsd = np.sqrt(np.mean(np.sum(delta**2, axis=1)))

delta_e = e_after - e_before
print(f"\n  {'Metric':<48s} {'Before':>12s} {'After':>12s}")
print("  " + "-" * 72)
print(f"  {'Potential energy (kJ/mol)':<48s} {e_before:>12.1f} {e_after:>12.1f}")
print(f"  {'Delta E (kJ/mol)':<48s} {'--':>12s} {delta_e:>12.1f}")
print(f"  {'Heavy-atom RMSD (A)':<48s} {'--':>12s} {rmsd:>12.3f}")
print(f"  {'Wall time (s)':<48s} {'--':>12s} {wall_time:>12.1f}")

# Save minimised structure
min_path = str(INTERMEDIATE_DIR / "08_minimised.pdb")
PDBFile.writeFile(
    simulation.topology, state_after.getPositions(), open(min_path, "w")
)

final_path = "/content/apo_protein_clean.pdb"
shutil.copy2(min_path, final_path)

final_size_kb = os.path.getsize(final_path) / 1024
print(f"\n  Minimised structure -> {min_path}")
print(f"  Final output -> {final_path}  ({final_size_kb:.0f} KB)")


# ---------------------------------------------------------------------------
#  STEP 9 : Final Validation and Docking Readiness Checklist
# ---------------------------------------------------------------------------

_print_header(9, "Final Validation and Docking Readiness Checklist")

# Parse final structure for statistics
struct_final = parser.get_structure("final", final_path)
model_final = list(struct_final.get_models())[0]
chains_final  = [c.id for c in model_final.get_chains()]
res_final     = [r for r in model_final.get_residues() if r.id[0] == " "]
het_final     = [r for r in model_final.get_residues() if r.id[0] not in (" ", "W")]
wat_final     = [r for r in model_final.get_residues() if r.id[0] == "W"]
atoms_final   = list(model_final.get_atoms())
h_final       = [a for a in atoms_final if a.element == "H"]
altloc_final  = [a for a in atoms_final if a.get_altloc() not in (" ", "")]

print(f"  {'Metric':<44s} {'Raw':>8s} {'Final':>8s}")
print("  " + "-" * 60)
print(f"  {'Standard AA residues':<44s} {len(residues):>8d} {len(res_final):>8d}")
print(f"  {'HETATM / ligand residues':<44s} {len(hetatm_res):>8d} {len(het_final):>8d}")
print(f"  {'Water molecules':<44s} {len(waters):>8d} {len(wat_final):>8d}")
print(f"  {'Total atoms':<44s} {len(atoms):>8d} {len(atoms_final):>8d}")
print(f"  {'Hydrogen atoms':<44s} {'0':>8s} {len(h_final):>8d}")
print(f"  {'Alt-conformation atoms':<44s} {len(altloc_atoms):>8d} {len(altloc_final):>8d}")

# Checklist
checks = {
    "Single model only":                  len(list(struct_final.get_models())) == 1,
    "All waters removed":                 len(wat_final) == 0,
    "All ligands removed (APO)":          len(het_final) == 0,
    "No alternate conformers":            len(altloc_final) == 0,
    "Hydrogens present":                  len(h_final) > 0,
    "pH-aware protonation (PropKa)":      True,
    "H-bond network optimised (pdb2pqr)": True,
    "Disulfide bonds handled":            True,
    "Energy minimised (AMBER14+GBN2)":    True,
    "Protein residues intact":            len(res_final) == len(residues),
}

print("\n  DOCKING READINESS CHECKLIST")
print("  " + "-" * 60)
all_pass = True
for label, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}]  {label}")

print(f"\n  Chains: {chains_final}  |  Residues: {len(res_final)}  |  Atoms: {len(atoms_final)}")

if all_pass:
    print("\n  Protein is READY for molecular docking.")
    print("  Next step: convert to PDBQT (e.g. mk_prepare_receptor from Meeko)")
else:
    print("\n  WARNING: Some checks failed. Review the output above.")


# ---------------------------------------------------------------------------
#  STEP 10 : Save Final Output and Download Intermediates
# ---------------------------------------------------------------------------

_print_header(10, "Download All Intermediate Files")

_files = [
    ("01_raw_input.pdb",           "Raw structure as fetched/uploaded"),
    ("02_chain_selected.pdb",      "After chain selection"),
    ("03_pdbfixer_apo.pdb",        "After PDBFixer (APO, heavy atoms only)"),
    ("04_biopy_cleaned.pdb",       "After alt-conformer and insertion-code cleanup"),
    ("05_propka_report.pka",       "PropKa pKa prediction report"),
    ("06_protonated_hbond.pdb",    "After pdb2pqr protonation + H-bond optimisation"),
    ("07_unminimised.pdb",         "Protonated structure BEFORE energy minimisation"),
    ("08_minimised.pdb",           "After energy minimisation"),
]

# Copy final output with a clear name
final_tagged = str(INTERMEDIATE_DIR / "09_apo_protein_clean_FINAL.pdb")
shutil.copy2(final_path, final_tagged)

print(f"\n  {'File':<48s} {'Size':>8s}   Description")
print("  " + "-" * 90)

for fname, desc in _files:
    fpath = INTERMEDIATE_DIR / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        print(f"  {fname:<48s} {size_kb:>6.0f} KB   {desc}")
    else:
        print(f"  {fname:<48s}      --   {desc} (not generated)")

print(f"  {'09_apo_protein_clean_FINAL.pdb':<48s} {os.path.getsize(final_tagged)/1024:>6.0f} KB   "
      f"FINAL output -- use this for docking")

# Trigger downloads in Colab
from google.colab import files as colab_files

for fname, _ in _files:
    fpath = INTERMEDIATE_DIR / fname
    if fpath.exists():
        colab_files.download(str(fpath))

colab_files.download(final_tagged)


# ---------------------------------------------------------------------------
#  Summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 72)
print("  PIPELINE COMPLETE")
print("=" * 72)
print(f"  Input         : {upload_name if structure_source == 'Upload PDB file' else pdb_id}")
print(f"  Final output  : apo_protein_clean.pdb")
print(f"  Intermediates : {INTERMEDIATE_DIR}  ({len(list(INTERMEDIATE_DIR.glob('*')))} files)")
print(f"  Pipeline      : PDBFixer -> BioPython -> PropKa -> pdb2pqr -> OpenMM EM")
print(f"  Force field   : AMBER14 + GBN2 implicit solvent")
print(f"  pH            : {target_pH}")
print(f"  EM restraint  : {restraint_k} kJ/mol/nm^2 on heavy atoms")
print(f"  Next step     : mk_prepare_receptor (Meeko) -> .pdbqt -> Vina")
print("=" * 72)

  STEP 0 | Installing Dependencies
  [OK]  openmm                   installed
  [OK]  pdbfixer                 installed
  [OK]  biopython                installed
  [OK]  propka                   installed
  [OK]  pdb2pqr                  installed
  [OK]  requests                 already installed
  [OK]  numpy                    already installed
  All dependencies ready.


  STEP 1 | Protein Input


Saving 3MXF.pdb to 3MXF.pdb
  Uploaded '3MXF.pdb' -> /content/prep_intermediates/01_raw_input.pdb

  STEP 2 | Initial Structure Audit
  Models                                   1
  Chains                                   ['A']
  Standard AA residues                     127
  HETATM residues                          6
  Water molecules                          208
  Total atoms                              1315
  Alt-conformation atoms                   0
  Residues with insertion codes            0

  Scanning for disulfide bonds (CYS Sg distance < 2.5 A) ...
  No disulfide bonds detected.

  B-factor distribution (Ca only) ...
    Mean   : 9.6 A^2
    Median : 8.6 A^2
    Std    : 4.3 A^2
    Max    : 42.8 A^2

  STEP 3 | Chain Selection
  Chains available: ['A']
  Single chain 'A' -- no selection needed.
  Chain-selected structure -> /content/prep_intermediates/02_chain_selected.pdb

  STEP 4 | PDBFixer: Gaps, Missing Heavy Atoms, APO Conversion
  No sequence gaps.
  No non-standard

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  PIPELINE COMPLETE
  Input         : 3MXF.pdb
  Final output  : apo_protein_clean.pdb
  Intermediates : /content/prep_intermediates  (11 files)
  Pipeline      : PDBFixer -> BioPython -> PropKa -> pdb2pqr -> OpenMM EM
  Force field   : AMBER14 + GBN2 implicit solvent
  pH            : 7.4
  EM restraint  : 500 kJ/mol/nm^2 on heavy atoms
  Next step     : mk_prepare_receptor (Meeko) -> .pdbqt -> Vina


# **Step 2: Ligand Preparation**

In [ ]:

!pip install rdkit

from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import files

# Upload original 3MXF.pdb
uploaded = files.upload()
pdb_filename = list(uploaded.keys())[0]

# Extract HETATM records (ligand) from PDB
hetatm_lines = []
with open(pdb_filename, 'r') as f:
    for line in f:
        if line.startswith('HETATM') and 'HOH' not in line:
            hetatm_lines.append(line)

# Print what ligands are found
residue_names = set(line[17:20].strip() for line in hetatm_lines)
print(f"Ligands found: {residue_names}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 40.7 MB/s eta 0:00:00


Saving 3MXF.pdb to 3MXF.pdb
Ligands found: {'EDO', 'JQ1', 'DMS', 'IOD'}


In [ ]:
# Extract JQ1 ligand atoms only
jq1_lines = [l for l in hetatm_lines if l[17:20].strip() == 'JQ1']
with open('xtal_ligand.pdb', 'w') as f:
    for line in jq1_lines:
        f.write(line)
    f.write('END\n')
print(f"JQ1 atoms found: {len(jq1_lines)}")

JQ1 atoms found: 31


In [1]:
!apt-get install -y openbabel -q

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7
The following NEW packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7 openbabel
0 upgraded, 5 newly installed, 0 to remove and 2 not upgraded.
Need to get 4,148 kB of archives.
After this operation, 19.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-iostreams1.74.0 amd64 1.74.0-14ubuntu3 [245 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libinchi1 amd64 1.03+dfsg-4 [455 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libmaeparser1 amd64 1.2.4-1build1 [88.2 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenbabel7 amd64 3.1.1+dfsg-6ubuntu5 [3,231 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openbabel amd64 3.1.1+dfsg-6

In [ ]:
import subprocess
from google.colab import files

result = subprocess.run(['obabel', 'xtal_ligand.pdb', '-O', 'xtal_ligand.sdf'],
               capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

print("✅ xtal_ligand.sdf saved!")
files.download('xtal_ligand.sdf')


1 molecule converted

✅ xtal_ligand.sdf saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =============================================================================
#  Ligand Preparation Pipeline for Molecular Docking
# =============================================================================
#  Pipeline : Sanitise -> Standardise -> Protonate -> Embed -> MMFF94s Minimise
#  Output   : Single lowest-energy 3D conformer per ligand in SDF/MOL2/PDB
# =============================================================================

# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

output_filename = "crystallized_ligand"  # @param {type:"string"}

protonation_method = "RDKit (neutralise charges)"  # @param ["OpenBabel (pH-aware)", "RDKit (neutralise charges)"]
target_pH = 7.4  # @param {type:"number"}
n_conformers = 5  # @param {type:"integer"}
mmff_max_iters = 2000  # @param {type:"integer"}
output_sdf = True  # @param {type:"boolean"}
output_mol2 = False  # @param {type:"boolean"}
output_pdb = False  # @param {type:"boolean"}

# ---------------------------------------------------------------------------
#  Resolve parameter values
# ---------------------------------------------------------------------------

OUTPUT_BASENAME = output_filename.strip().replace(" ", "_") + "_clean"
PROTONATION_METHOD = "openbabel" if "OpenBabel" in protonation_method else "rdkit"
TARGET_PH = target_pH
N_CONFORMERS = n_conformers
MMFF_MAX_ITERS = mmff_max_iters
OUT_SDF = output_sdf
OUT_MOL2 = output_mol2
OUT_PDB = output_pdb

if not any([OUT_SDF, OUT_MOL2, OUT_PDB]):
    raise ValueError("Please select at least one output format.")

# ---------------------------------------------------------------------------
#  STEP 0 : Install Dependencies
# ---------------------------------------------------------------------------

import sys, subprocess

def _print_header(step_num, title):
    print("\n" + "=" * 72)
    print(f"  STEP {step_num} | {title}")
    print("=" * 72)

print("=" * 72)
print("  STEP 0 | Installing Dependencies")
print("=" * 72)

def _run(cmd):
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for label, pip_name, import_name in [
    ("rdkit", "rdkit", "rdkit.Chem"),
    ("tqdm", "tqdm", "tqdm"),
]:
    try:
        __import__(import_name)
        print(f"  [OK]  {label:24s} already installed")
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        print(f"  [OK]  {label:24s} installed")

try:
    import openbabel
    print(f"  [OK]  {'openbabel':24s} already installed")
except ImportError:
    _run(["apt-get", "install", "-y", "-q", "openbabel"])
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openbabel-wheel"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    print(f"  [OK]  {'openbabel':24s} installed")

print("  All dependencies ready.\n")

# ---------------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------------

import os, time, warnings, tempfile
from copy import deepcopy
from pathlib import Path
from tqdm import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdDistGeom
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import SDWriter
from openbabel import openbabel as ob

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

print("  Configuration")
print("  " + "-" * 60)
print(f"  Output basename  : {OUTPUT_BASENAME}")
print(f"  Protonation      : {PROTONATION_METHOD.upper()}"
      + (f" at pH {TARGET_PH}" if PROTONATION_METHOD == "openbabel" else ""))
selected_labels = []
if OUT_SDF:  selected_labels.append("SDF")
if OUT_MOL2: selected_labels.append("MOL2")
if OUT_PDB:  selected_labels.append("PDB")
print(f"  Output formats   : {', '.join(selected_labels)}")
print(f"  Conformers       : {N_CONFORMERS} (internal; lowest-energy kept)")
print(f"  MMFF94s max iter : {MMFF_MAX_ITERS}")

# ---------------------------------------------------------------------------
#  STEP 1 : File Upload
# ---------------------------------------------------------------------------

_print_header(1, "File Upload")
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file uploaded. Please re-run the cell.")

upload_name = list(uploaded.keys())[0]
upload_path = Path(upload_name)
ext = upload_path.suffix.lower()

if ext not in (".sdf", ".mol2", ".pdb"):
    raise ValueError(f"Unsupported format '{ext}'. Please upload .sdf, .mol2, or .pdb")

print(f"\n  Uploaded : {upload_name}  ({len(uploaded[upload_name]):,} bytes)")
print(f"  Format   : {ext.upper()}")

# ---------------------------------------------------------------------------
#  STEP 2 : File Inspection and Parsing
# ---------------------------------------------------------------------------

_print_header(2, "File Inspection and Parsing")

def load_sdf(path):
    mols = []
    supplier = Chem.SDMolSupplier(str(path), removeHs=True, sanitize=True)
    for i, mol in enumerate(supplier):
        if mol is None: continue
        name = mol.GetPropsAsDict().get("_Name", "").strip() or f"mol_{i+1:04d}"
        mol.SetProp("_Name", name)
        mols.append((mol, name))
    return mols

def load_mol2(path):
    mols = []
    raw = path.read_text()
    blocks = ["@<TRIPOS>MOLECULE" + b for b in raw.split("@<TRIPOS>MOLECULE") if b.strip()]
    for i, block in enumerate(blocks):
        mol = Chem.MolFromMol2Block(block, removeHs=True, sanitize=True)
        if mol is None: continue
        lines = block.strip().splitlines()
        name = (lines[1].strip() if len(lines) > 1 else "") or f"mol_{i+1:04d}"
        mol.SetProp("_Name", name)
        mols.append((mol, name))
    return mols

def load_pdb(path):
    mols = []
    raw = path.read_text()
    blocks = []
    if "MODEL" in raw:
        current = []
        for line in raw.splitlines():
            if line.startswith("MODEL"): current = [line]
            elif line.startswith("ENDMDL"):
                current.append(line)
                blocks.append("\n".join(current))
                current = []
            else: current.append(line)
        if not blocks: blocks = [raw]
    else: blocks = [raw]

    for i, block in enumerate(blocks):
        mol = Chem.MolFromPDBBlock(block, removeHs=True, sanitize=True)
        if mol is None: continue
        name = f"mol_{i+1:04d}"
        for line in block.splitlines():
            if line.startswith("COMPND") or line.startswith("REMARK"):
                candidate = line.split(None, 1)[-1].strip()
                if candidate:
                    name = candidate[:30]
                    break
        mol.SetProp("_Name", name)
        mols.append((mol, name))
    return mols

loaders = {".sdf": load_sdf, ".mol2": load_mol2, ".pdb": load_pdb}
raw_mols = loaders[ext](upload_path)
n_valid = len(raw_mols)
print(f"  Parseable molecules : {n_valid}")

if n_valid == 0:
    raise ValueError("No valid molecules found.")

# ---------------------------------------------------------------------------
#  STEP 3 : Ligand Preparation
# ---------------------------------------------------------------------------

_print_header(3, "Ligand Preparation")

def standardise_mol(mol):
    lfc = rdMolStandardize.LargestFragmentChooser()
    te = rdMolStandardize.TautomerEnumerator()
    mol = lfc.choose(mol)
    mol = te.Canonicalize(mol)
    return mol

def protonate_rdkit(mol):
    uc = rdMolStandardize.Uncharger()
    return uc.uncharge(mol)

def protonate_openbabel(mol, ph=7.4):
    try:
        smi = Chem.MolToSmiles(mol)
        obconv = ob.OBConversion()
        obconv.SetInAndOutFormats("smi", "smi")
        obmol = ob.OBMol()
        obconv.ReadString(obmol, smi)
        obmol.AddHydrogens(False, True, ph)
        out_smi = obconv.WriteString(obmol).strip()
        out_mol = Chem.MolFromSmiles(out_smi)
        if out_mol is None: return mol
        out_mol.SetProp("_Name", mol.GetPropsAsDict().get("_Name", ""))
        return out_mol
    except: return mol

def embed_and_minimise(mol, n_confs, max_iters):
    params = rdDistGeom.ETKDGv3()
    params.randomSeed = 42
    mol3d = Chem.AddHs(mol)
    conf_ids = AllChem.EmbedMultipleConfs(mol3d, numConfs=n_confs, params=params)
    if not conf_ids: AllChem.EmbedMolecule(mol3d, AllChem.ETKDG())

    results = AllChem.MMFFOptimizeMoleculeConfs(mol3d, mmffVariant="MMFF94s", maxIters=max_iters)
    if not results: return mol3d, False

    energies = [(r[1], i) for i, r in enumerate(results) if r[0] == 0]
    best_conf = min(energies)[1] if energies else 0

    mol_out = deepcopy(mol3d)
    for cid in reversed(range(mol_out.GetNumConformers())):
        if cid != best_conf: mol_out.RemoveConformer(cid)
    return mol_out, bool(energies)

prepared = []
failed = []

for idx, (raw_mol, mol_name) in enumerate(tqdm(raw_mols, desc="  Preparing")):
    try:
        mol = Chem.AddHs(raw_mol)
        Chem.SanitizeMol(mol)
        mol = standardise_mol(mol)

        if PROTONATION_METHOD == "openbabel":
            mol = protonate_openbabel(mol, ph=TARGET_PH)
        else:
            mol = protonate_rdkit(mol)

        mol = Chem.AddHs(mol)
        mol3d, converged = embed_and_minimise(mol, n_confs=N_CONFORMERS, max_iters=MMFF_MAX_ITERS)

        mol3d.SetProp("_Name", mol_name)
        mol3d.SetProp("Converged", str(converged))
        prepared.append((mol3d, mol_name))
    except Exception as e:
        failed.append((mol_name, str(e)))

# ---------------------------------------------------------------------------
#  STEP 4 & 5 : Summary & Output
# ---------------------------------------------------------------------------

_print_header(4, "Summary & Writing Files")
print(f"  Success: {len(prepared)} | Failed: {len(failed)}")

if OUT_SDF:
    out_path = f"/content/{OUTPUT_BASENAME}.sdf"
    with SDWriter(out_path) as w:
        for m, n in prepared: w.write(m)
    print(f"  [OK] Saved SDF to {out_path}")
    files.download(out_path)

print("\n" + "=" * 72)
print("  PIPELINE COMPLETE - RDKIT NEUTRALISATION MODE")
print("=" * 72)

  STEP 0 | Installing Dependencies
  [OK]  rdkit                    already installed
  [OK]  tqdm                     already installed
  [OK]  openbabel                already installed
  All dependencies ready.

  Configuration
  ------------------------------------------------------------
  Output basename  : crystallized_ligand_clean
  Protonation      : RDKIT
  Output formats   : SDF
  Conformers       : 5 (internal; lowest-energy kept)
  MMFF94s max iter : 2000

  STEP 1 | File Upload


Saving xtal_ligand.sdf to xtal_ligand (2).sdf

  Uploaded : xtal_ligand (2).sdf  (3,010 bytes)
  Format   : .SDF

  STEP 2 | File Inspection and Parsing
  Parseable molecules : 1

  STEP 3 | Ligand Preparation


  Preparing: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s]


  STEP 4 | Summary & Writing Files
  Success: 1 | Failed: 0
  [OK] Saved SDF to /content/crystallized_ligand_clean.sdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  PIPELINE COMPLETE - RDKIT NEUTRALISATION MODE


# **Step 3a: Validation Studies using Cognate Docking**

In [ ]:
# =============================================================================
#  AutoDock Vina -- Cognate Docking Validation (Redocking + RMSD)
# =============================================================================
#  Workflow : Upload protein + co-crystallised ligand + clean ligand
#             -> define box from co-crystal -> dock clean ligand -> RMSD
#  Output   : Docked poses (SDF/PDBQT), RMSD table (CSV), 3D visualisation
#
#  Usage    : Run this cell in Google Colab. Adjust parameters in the sidebar.
# =============================================================================


# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

# @markdown ### Binding-Site Box
# @markdown Box is centred on the co-crystallised ligand with the specified padding.
REF_LIGAND_PADDING = 6.0  # @param {type:"number"}

# @markdown ---
# @markdown ### Docking Engine
SCORING_FUNCTION = "vinardo"  # @param ["vina", "vinardo"]
EXHAUSTIVENESS   = 8          # @param {type:"integer"}
N_POSES          = 1          # @param {type:"integer"}
CPU_CORES        = 0          # @param {type:"integer"}
RANDOM_SEED      = 42         # @param {type:"integer"}

# @markdown ---
# @markdown ### Output Options
RUN_NAME         = "cognate_validation"  # @param {type:"string"}
EXPORT_POSE_PDBS = True                  # @param {type:"boolean"}

# @markdown ---
# @markdown ### RMSD Settings
# @markdown Symmetry-corrected RMSD uses the Hungarian algorithm to handle
# @markdown equivalent atoms (e.g. phenyl ring flips).
SYMMETRY_CORRECTED_RMSD = True  # @param {type:"boolean"}
RMSD_SUCCESS_THRESHOLD  = 2.0   # @param {type:"number"}


# ---------------------------------------------------------------------------
#  Resolve form values
# ---------------------------------------------------------------------------

import subprocess, sys, os, shutil, time, zipfile, csv, re
from pathlib import Path

pad_ref        = REF_LIGAND_PADDING
scoring_fn     = SCORING_FUNCTION.strip().lower()
exhaustiveness = EXHAUSTIVENESS
n_poses        = N_POSES
cpu_cores      = CPU_CORES
seed           = RANDOM_SEED
run_name       = re.sub(r"[^\w\-]", "_", RUN_NAME) if RUN_NAME else "cognate_validation"
export_pose_pdbs  = EXPORT_POSE_PDBS
symmetry_rmsd     = SYMMETRY_CORRECTED_RMSD
rmsd_threshold    = RMSD_SUCCESS_THRESHOLD


# ---------------------------------------------------------------------------
#  STEP 1 : Install Dependencies
# ---------------------------------------------------------------------------

def _pip(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    return r.returncode == 0

def _print_header(step, total, title):
    print("\n" + "=" * 72)
    print(f"  STEP {step}/{total}  |  {title}")
    print("=" * 72)

_print_header(1, 7, "Installing Dependencies")

_PKGS = {
    "vina":            "AutoDock Vina 1.2.x Python bindings",
    "openbabel-wheel": "Format conversion (PDB/SDF/MOL2 -> PDBQT)",
    "scipy":           "Hungarian algorithm for symmetry-corrected RMSD",
}
for pkg, desc in _PKGS.items():
    status = "installed" if _pip(pkg) else "FAILED"
    print(f"  [{'OK' if status == 'installed' else '!!'}]  {pkg:<22s} {desc}")

try:
    from rdkit import Chem
    print(f"  [OK]  {'rdkit':<22s} already available")
except ImportError:
    _pip("rdkit-pypi")
    print(f"  [OK]  {'rdkit':<22s} installed")

print("  All dependencies ready.")


# ---------------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------------

import numpy as np
from google.colab import files
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, rdMolAlign
from openbabel import openbabel as ob


# ---------------------------------------------------------------------------
#  Working directory
# ---------------------------------------------------------------------------

WORKDIR = Path("/content/vina_docking")
WORKDIR.mkdir(exist_ok=True)


# ---------------------------------------------------------------------------
#  STEP 2 : File Upload
# ---------------------------------------------------------------------------

_print_header(2, 7, "File Upload")

# -- Protein (cleaned APO structure) ----------------------------------------
print("  Protein (.pdb) -- cleaned, ligand/water removed:")
prot_up   = files.upload()
prot_orig = list(prot_up.keys())[0]
if not prot_orig.lower().endswith(".pdb"):
    raise ValueError(f"Protein must be .pdb -- got '{prot_orig}'")
prot_in = WORKDIR / "input_protein.pdb"
shutil.copy(prot_orig, prot_in)
print(f"  [OK]  {prot_orig}  ({prot_in.stat().st_size / 1024:.1f} KB)")

# -- Co-crystallised ligand (reference pose = ground truth) ------------------
print("\n  Co-crystallised ligand (.sdf / .mol2 / .pdb) -- the crystal pose:")
cocrys_up   = files.upload()
cocrys_orig = list(cocrys_up.keys())[0]
cocrys_ext  = Path(cocrys_orig).suffix.lower()
if cocrys_ext not in (".sdf", ".mol2", ".pdb"):
    raise ValueError(f"Co-crystallised ligand must be .sdf/.mol2/.pdb -- got '{cocrys_orig}'")
cocrys_path = WORKDIR / f"cocrystal_ligand{cocrys_ext}"
shutil.copy(cocrys_orig, cocrys_path)
print(f"  [OK]  {cocrys_orig}  ({cocrys_path.stat().st_size / 1024:.1f} KB)")

# -- Clean ligand (to be redocked) ------------------------------------------
print("\n  Clean ligand (.sdf / .mol2) -- will be redocked:")
print("  (Same molecule, fresh 3D conformer or energy-minimised)")
clean_up   = files.upload()
clean_orig = list(clean_up.keys())[0]
clean_ext  = Path(clean_orig).suffix.lower()
if clean_ext not in (".sdf", ".mol2"):
    raise ValueError(f"Clean ligand must be .sdf or .mol2 -- got '{clean_orig}'")
clean_path = WORKDIR / f"clean_ligand{clean_ext}"
shutil.copy(clean_orig, clean_path)
print(f"  [OK]  {clean_orig}  ({clean_path.stat().st_size / 1024:.1f} KB)")


# ---------------------------------------------------------------------------
#  STEP 3 : Parse Co-crystallised Ligand and Define Docking Box
# ---------------------------------------------------------------------------

_print_header(3, 7, "Parsing Co-crystallised Ligand and Defining Box")


def _load_mol(filepath):
    """Load a single molecule via RDKit, falling back to OpenBabel if needed."""
    ext = filepath.suffix.lower()
    mol = None

    if ext == ".sdf":
        mol = next((m for m in Chem.SDMolSupplier(str(filepath), removeHs=False)
                     if m is not None), None)
    elif ext == ".mol2":
        mol = Chem.MolFromMol2File(str(filepath), removeHs=False)
    elif ext == ".pdb":
        mol = Chem.MolFromPDBFile(str(filepath), removeHs=False)

    # OpenBabel fallback if RDKit fails
    if mol is None:
        print(f"  WARNING: RDKit could not parse {ext} -- trying OpenBabel ...")
        try:
            oc = ob.OBConversion()
            oc.SetInAndOutFormats(ext.lstrip("."), "sdf")
            om = ob.OBMol()
            oc.ReadFile(om, str(filepath))
            tmp = filepath.parent / f"{filepath.stem}_conv.sdf"
            oc.WriteFile(om, str(tmp))
            mol = next((m for m in Chem.SDMolSupplier(str(tmp), removeHs=False)
                        if m is not None), None)
            if mol:
                print("  [OK]  OpenBabel conversion succeeded")
        except Exception as e:
            print(f"  WARNING: OpenBabel also failed: {e}")
    return mol


ref_mol = _load_mol(cocrys_path)
if ref_mol is None:
    raise RuntimeError("Could not parse co-crystallised ligand. Check file.")
if ref_mol.GetNumConformers() == 0 or not ref_mol.GetConformer().Is3D():
    raise RuntimeError("Co-crystallised ligand has no 3D coordinates.")

# Compute box centre (ligand centroid) and dimensions
ref_pos = np.array([ref_mol.GetConformer().GetAtomPosition(i)
                     for i in range(ref_mol.GetNumAtoms())])
center = ref_pos.mean(axis=0).tolist()
size   = (ref_pos.max(axis=0) - ref_pos.min(axis=0) + 2 * pad_ref).clip(min=15).tolist()

n_heavy_ref = ref_mol.GetNumHeavyAtoms()
ref_formula = rdMolDescriptors.CalcMolFormula(ref_mol)

print(f"  Co-crystallised ligand : {ref_formula}")
print(f"  Atoms                  : {ref_mol.GetNumAtoms()} total ({n_heavy_ref} heavy)")
print(f"  Centroid               : ({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}) A")
print(f"  Box size               : ({size[0]:.2f}, {size[1]:.2f}, {size[2]:.2f}) A  "
      f"[padding = {pad_ref} A]")
print(f"  Box volume             : {size[0] * size[1] * size[2]:,.0f} A^3")


# ---------------------------------------------------------------------------
#  STEP 4 : Protein PDB -> PDBQT
# ---------------------------------------------------------------------------

_print_header(4, 7, "Protein Conversion (PDB -> PDBQT)")

# -- Parse coordinates for sanity check -------------------------------------
print("  [4a] Parsing coordinates ...")
all_coords, res_coords = [], {}
with open(prot_in) as fh:
    for line in fh:
        if line[:4] != "ATOM":
            continue
        try:
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            rnum = int(line[22:26])
            all_coords.append([x, y, z])
            res_coords.setdefault(rnum, []).append([x, y, z])
        except (ValueError, IndexError):
            continue

if not all_coords:
    raise RuntimeError("No ATOM records in PDB. Check input file.")

print(f"       {len(all_coords):,} atoms, {len(res_coords)} residues")

# -- Convert PDB -> PDBQT via OpenBabel ------------------------------------
print("  [4b] PDB -> PDBQT (OpenBabel, Gasteiger charges, rigid) ...")
obC = ob.OBConversion()
obC.SetInAndOutFormats("pdb", "pdbqt")
obC.AddOption("r", ob.OBConversion.OUTOPTIONS)  # rigid receptor

prot_mol = ob.OBMol()
if not obC.ReadFile(prot_mol, str(prot_in)):
    raise RuntimeError("OpenBabel could not read protein PDB.")

cm = ob.OBChargeModel.FindType("gasteiger")
if cm:
    cm.ComputeCharges(prot_mol)

prot_pdbqt = WORKDIR / "protein.pdbqt"
obC.WriteFile(prot_mol, str(prot_pdbqt))
n_rec = sum(1 for ln in open(prot_pdbqt) if ln.startswith(("ATOM", "HETATM")))
print(f"       {n_rec:,} PDBQT records -> protein.pdbqt")


# ---------------------------------------------------------------------------
#  STEP 5 : Parse Clean Ligand
# ---------------------------------------------------------------------------

_print_header(5, 7, "Parsing Clean Ligand")

if clean_ext == ".sdf":
    clean_mol = next((m for m in Chem.SDMolSupplier(str(clean_path),
                                                     removeHs=False, sanitize=True)
                      if m is not None), None)
elif clean_ext == ".mol2":
    # Convert MOL2 -> SDF via OpenBabel, then read with RDKit
    obW = ob.OBConversion()
    obW.SetInAndOutFormats("mol2", "sdf")
    _m = ob.OBMol()
    obW.ReadFile(_m, str(clean_path))
    clean_sdf_conv = WORKDIR / "clean_from_mol2.sdf"
    obW.WriteFile(_m, str(clean_sdf_conv))
    clean_mol = next((m for m in Chem.SDMolSupplier(str(clean_sdf_conv),
                                                     removeHs=False, sanitize=True)
                      if m is not None), None)

if clean_mol is None:
    raise RuntimeError("Could not parse clean ligand. Check file.")
if clean_mol.GetNumConformers() == 0 or not clean_mol.GetConformer().Is3D():
    raise RuntimeError("Clean ligand has no 3D coordinates.")

# -- Molecular descriptors --------------------------------------------------
mw      = Descriptors.ExactMolWt(clean_mol)
logp    = Descriptors.MolLogP(clean_mol)
hbd     = rdMolDescriptors.CalcNumHBD(clean_mol)
hba     = rdMolDescriptors.CalcNumHBA(clean_mol)
rotb    = rdMolDescriptors.CalcNumRotatableBonds(clean_mol)
tpsa    = Descriptors.TPSA(clean_mol)
formula = rdMolDescriptors.CalcMolFormula(clean_mol)
ro5_ok  = sum([mw <= 500, logp <= 5, hbd <= 5, hba <= 10]) >= 3

lig_name_raw = clean_mol.GetPropsAsDict().get("_Name", "").strip()
lig_name = re.sub(r"[^\w\-]", "_", lig_name_raw) if lig_name_raw else "LIGAND"

print(f"  Name         : {lig_name}")
print(f"  Formula      : {formula}   MW = {mw:.1f} Da")
print(f"  LogP = {logp:.2f}   HBD = {hbd}   HBA = {hba}   RotB = {rotb}   TPSA = {tpsa:.0f} A^2")
print(f"  Lipinski Ro5 : {'PASS' if ro5_ok else 'FAIL'}")
print(f"  Atoms        : {clean_mol.GetNumAtoms()} total ({clean_mol.GetNumHeavyAtoms()} heavy)")
print(f"  3D coords    : confirmed")

# -- Sanity check: same molecule? -------------------------------------------
ref_smi   = Chem.MolToSmiles(Chem.RemoveHs(ref_mol))
clean_smi = Chem.MolToSmiles(Chem.RemoveHs(clean_mol))
if ref_smi == clean_smi:
    print(f"\n  [OK]  Canonical SMILES match -- same molecule confirmed")
else:
    print(f"\n  WARNING: SMILES differ between co-crystallised and clean ligands!")
    print(f"    Co-crystal : {ref_smi[:80]}{'...' if len(ref_smi) > 80 else ''}")
    print(f"    Clean      : {clean_smi[:80]}{'...' if len(clean_smi) > 80 else ''}")
    print(f"    Proceeding, but RMSD may not be meaningful if atoms differ.")


# ---------------------------------------------------------------------------
#  STEP 6 : Settings Confirmation
# ---------------------------------------------------------------------------

_print_header(6, 7, "Settings Confirmation")

print(f"  Protein           : {prot_orig}")
print(f"  Co-crystal ligand : {cocrys_orig}  (reference pose for RMSD)")
print(f"  Clean ligand      : {clean_orig}  (to be redocked)")
print(f"  Ligand name       : {lig_name}  {formula}")
print(f"  Box centre        : ({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}) A")
print(f"  Box size          : ({size[0]:.2f}, {size[1]:.2f}, {size[2]:.2f}) A")
print(f"  Box padding       : {pad_ref} A")
print(f"  Scoring function  : {scoring_fn}")
print(f"  Exhaustiveness    : {exhaustiveness}")
print(f"  N poses           : {n_poses}")
print(f"  CPUs              : {'auto' if cpu_cores == 0 else cpu_cores}")
print(f"  Seed              : {seed}")
print(f"  Symmetry RMSD     : {'Yes' if symmetry_rmsd else 'No'}")
print(f"  Success threshold : {rmsd_threshold} A")
print(f"  Run name          : {run_name}")
print(f"\n  Settings locked.")


# ---------------------------------------------------------------------------
#  STEP 7 : Docking + RMSD Calculation
# ---------------------------------------------------------------------------

_print_header(7, 7, "Redocking and RMSD Calculation")

from vina import Vina


def _ob_sdf_to_pdbqt(sdf_path, pdbqt_path):
    """Convert a single-molecule SDF to PDBQT via OpenBabel."""
    obc = ob.OBConversion()
    obc.SetInAndOutFormats("sdf", "pdbqt")
    mol = ob.OBMol()
    if not obc.ReadFile(mol, str(sdf_path)):
        raise RuntimeError(f"OpenBabel could not read {sdf_path.name}")
    cm2 = ob.OBChargeModel.FindType("gasteiger")
    if cm2:
        cm2.ComputeCharges(mol)
    obc.WriteFile(mol, str(pdbqt_path))


def _pdbqt_to_sdf(pdbqt_path, sdf_path):
    """Convert docked PDBQT back to SDF via OpenBabel."""
    obc = ob.OBConversion()
    obc.SetInAndOutFormats("pdbqt", "sdf")
    mol = ob.OBMol()
    obc.ReadFile(mol, str(pdbqt_path))
    obc.WriteFile(mol, str(sdf_path))


# -- RMSD helper functions ---------------------------------------------------

def _heavy_atom_coords(mol):
    """Extract heavy-atom coordinates as (N, 3) array."""
    conf = mol.GetConformer()
    coords = []
    for i in range(mol.GetNumAtoms()):
        if mol.GetAtomWithIdx(i).GetAtomicNum() > 1:
            pos = conf.GetAtomPosition(i)
            coords.append([pos.x, pos.y, pos.z])
    return np.array(coords)


def _heavy_atom_elements(mol):
    """Extract heavy-atom element symbols."""
    return [mol.GetAtomWithIdx(i).GetSymbol()
            for i in range(mol.GetNumAtoms())
            if mol.GetAtomWithIdx(i).GetAtomicNum() > 1]


def rmsd_simple(coords1, coords2):
    """Plain RMSD assuming matched atom order."""
    if coords1.shape != coords2.shape:
        raise ValueError(f"Shape mismatch: {coords1.shape} vs {coords2.shape}")
    return np.sqrt(np.mean(np.sum((coords1 - coords2) ** 2, axis=1)))


def rmsd_symmetry_corrected(ref_mol_h, docked_mol_h):
    """
    Symmetry-corrected RMSD using RDKit GetBestRMS.
    Accounts for equivalent atoms (e.g. ring flips, symmetric groups).
    Falls back to Hungarian assignment if GetBestRMS fails.
    """
    try:
        return rdMolAlign.GetBestRMS(Chem.RemoveHs(ref_mol_h),
                                      Chem.RemoveHs(docked_mol_h))
    except Exception as e1:
        print(f"    WARNING: GetBestRMS failed ({e1}), trying Hungarian fallback ...")
        try:
            from scipy.optimize import linear_sum_assignment
            c1 = _heavy_atom_coords(ref_mol_h)
            c2 = _heavy_atom_coords(docked_mol_h)
            e1_list = _heavy_atom_elements(ref_mol_h)
            e2_list = _heavy_atom_elements(docked_mol_h)
            n = len(c1)
            if n != len(c2):
                raise ValueError(f"Different heavy-atom counts: {n} vs {len(c2)}")
            # Cost matrix: squared distances, with large penalty for element mismatches
            cost = np.full((n, n), 1e12)
            for i in range(n):
                for j in range(n):
                    if e1_list[i] == e2_list[j]:
                        cost[i, j] = np.sum((c1[i] - c2[j]) ** 2)
            row_ind, col_ind = linear_sum_assignment(cost)
            return np.sqrt(np.mean(cost[row_ind, col_ind]))
        except Exception as e2:
            print(f"    WARNING: Hungarian fallback also failed ({e2})")
            return None


# -- Prepare clean ligand PDBQT ----------------------------------------------
lig_dir = WORKDIR / "ligand_dock"
lig_dir.mkdir(exist_ok=True)

clean_sdf = lig_dir / "clean_ligand.sdf"
w = Chem.SDWriter(str(clean_sdf))
w.write(clean_mol)
w.close()

lig_pdbqt = lig_dir / "clean_ligand.pdbqt"
_ob_sdf_to_pdbqt(clean_sdf, lig_pdbqt)
print(f"  Clean ligand SDF -> PDBQT  (RotB = {rotb})")

# -- Run Vina ----------------------------------------------------------------
print(f"  Running Vina (exhaustiveness = {exhaustiveness}, poses = {n_poses}) ...")

v = Vina(sf_name=scoring_fn,
         cpu=cpu_cores if cpu_cores > 0 else 0,
         seed=seed,
         verbosity=0)
v.set_receptor(str(prot_pdbqt))
v.set_ligand_from_file(str(lig_pdbqt))
v.compute_vina_maps(center=center, box_size=size)

t0 = time.time()
v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
elapsed = time.time() - t0

energies = v.energies(n_poses=n_poses)
print(f"  Done in {elapsed:.1f}s -- {len(energies)} pose(s) generated\n")

# -- Save docked poses -------------------------------------------------------
out_pdbqt = lig_dir / "docked_poses.pdbqt"
out_sdf   = lig_dir / "docked_poses.sdf"
v.write_poses(str(out_pdbqt), n_poses=n_poses, overwrite=True)

_pdbqt_to_sdf(out_pdbqt, out_sdf)
docked_mols = [m for m in Chem.SDMolSupplier(str(out_sdf), removeHs=False)
               if m is not None]
print(f"  {len(docked_mols)} pose(s) written to SDF\n")

# -- Compute RMSD for every pose ---------------------------------------------
print("  RMSD Calculation (reference = co-crystallised pose)")
if symmetry_rmsd:
    print("  Method: symmetry-corrected (RDKit GetBestRMS)\n")
else:
    print("  Method: simple heavy-atom RMSD (matched atom order)\n")

ref_heavy_coords = _heavy_atom_coords(ref_mol)
rmsd_results = []

print(f"  {'Pose':<6} {'Affinity':>10}  {'RMSD':>8}  {'RMSD_lb':>8}  {'RMSD_ub':>8}  {'Status'}")
print(f"  {'-' * 62}")

for i, docked_mol in enumerate(docked_mols):
    aff     = energies[i][0] if i < len(energies) else float('nan')
    rmsd_lb = energies[i][1] if i < len(energies) else float('nan')
    rmsd_ub = energies[i][2] if i < len(energies) else float('nan')

    # Compute RMSD vs crystal reference
    rms = None
    if symmetry_rmsd:
        rms = rmsd_symmetry_corrected(ref_mol, docked_mol)
    else:
        try:
            docked_heavy = _heavy_atom_coords(docked_mol)
            if ref_heavy_coords.shape == docked_heavy.shape:
                rms = rmsd_simple(ref_heavy_coords, docked_heavy)
            else:
                print(f"    WARNING: Pose {i + 1}: atom count mismatch "
                      f"({ref_heavy_coords.shape[0]} vs {docked_heavy.shape[0]})")
        except Exception as e:
            print(f"    WARNING: Pose {i + 1}: RMSD error -- {e}")

    status = ""
    if rms is not None:
        status = "PASS" if rms <= rmsd_threshold else "FAIL"

    rmsd_results.append({
        "pose":     i + 1,
        "affinity": aff,
        "rmsd":     rms,
        "rmsd_lb":  rmsd_lb,
        "rmsd_ub":  rmsd_ub,
        "status":   status,
    })

    rms_str = f"{rms:.3f}" if rms is not None else "N/A"
    tag = " <-- best" if i == 0 else ""
    print(f"  {i + 1:<6} {aff:>10.3f}  {rms_str:>8}  {rmsd_lb:>8.3f}  {rmsd_ub:>8.3f}  {status}{tag}")

# -- Summarise best poses ----------------------------------------------------
valid_rmsd = [r for r in rmsd_results if r["rmsd"] is not None]

if valid_rmsd:
    best_rmsd_row = min(valid_rmsd, key=lambda r: r["rmsd"])
    best_aff_row  = min(valid_rmsd, key=lambda r: r["affinity"])

    print(f"\n  {'-' * 62}")
    print(f"  Best RMSD     : Pose {best_rmsd_row['pose']}  "
          f"RMSD = {best_rmsd_row['rmsd']:.3f} A  "
          f"Affinity = {best_rmsd_row['affinity']:.3f} kcal/mol")
    print(f"  Best Affinity : Pose {best_aff_row['pose']}  "
          f"RMSD = {best_aff_row['rmsd']:.3f} A  "
          f"Affinity = {best_aff_row['affinity']:.3f} kcal/mol")

    if best_rmsd_row['rmsd'] <= rmsd_threshold:
        print(f"\n  VALIDATION PASSED -- Best RMSD ({best_rmsd_row['rmsd']:.3f} A) "
              f"<= {rmsd_threshold} A threshold")
    else:
        print(f"\n  VALIDATION FAILED -- Best RMSD ({best_rmsd_row['rmsd']:.3f} A) "
              f"> {rmsd_threshold} A threshold")

    if best_aff_row['pose'] == best_rmsd_row['pose']:
        print(f"  Rank-1 (best affinity) IS the best-RMSD pose.")
    else:
        print(f"  NOTE: Rank-1 (best affinity) is Pose {best_aff_row['pose']} "
              f"(RMSD = {best_aff_row['rmsd']:.3f} A); "
              f"best-RMSD is Pose {best_rmsd_row['pose']}.")

    n_pass = sum(1 for r in valid_rmsd if r["rmsd"] <= rmsd_threshold)
    print(f"  {n_pass}/{len(valid_rmsd)} pose(s) <= {rmsd_threshold} A threshold")
else:
    print("\n  WARNING: No valid RMSD values could be computed!")


# ---------------------------------------------------------------------------
#  Output: Annotated SDF
# ---------------------------------------------------------------------------

print("\n  Writing annotated docked_poses_with_rmsd.sdf ...")
annotated_sdf = lig_dir / "docked_poses_with_rmsd.sdf"
wrt = Chem.SDWriter(str(annotated_sdf))
for i, m in enumerate(docked_mols):
    if i < len(energies):
        m.SetProp("_Name",                  f"{lig_name}_Pose_{i + 1}")
        m.SetProp("Vina_Affinity_kcal_mol", f"{energies[i][0]:.4f}")
        m.SetProp("RMSD_lower_bound_A",     f"{energies[i][1]:.4f}")
        m.SetProp("RMSD_upper_bound_A",     f"{energies[i][2]:.4f}")
        m.SetProp("Rank",                   str(i + 1))
    if i < len(rmsd_results) and rmsd_results[i]["rmsd"] is not None:
        m.SetProp("RMSD_vs_crystal_A",      f"{rmsd_results[i]['rmsd']:.4f}")
        m.SetProp("RMSD_status",            rmsd_results[i]["status"].strip())
    wrt.write(m)
wrt.close()
print(f"  [OK]  {len(docked_mols)} pose(s) -> docked_poses_with_rmsd.sdf")


# ---------------------------------------------------------------------------
#  Output: Vina config file (for reproducibility)
# ---------------------------------------------------------------------------

ts = time.strftime("%Y-%m-%d %H:%M:%S")
with open(lig_dir / "vina_config.txt", "w") as fh:
    fh.write(f"# AutoDock Vina Config -- Cognate Docking Validation\n")
    fh.write(f"# {ts}\n")
    fh.write(f"# Protein: {prot_orig}  |  Co-crystal: {cocrys_orig}  |  Clean: {clean_orig}\n\n")
    fh.write(f"receptor = ../protein.pdbqt\n")
    fh.write(f"ligand   = clean_ligand.pdbqt\n")
    fh.write(f"out      = docked_poses.pdbqt\n\n")
    fh.write(f"center_x = {center[0]:.4f}\n")
    fh.write(f"center_y = {center[1]:.4f}\n")
    fh.write(f"center_z = {center[2]:.4f}\n")
    fh.write(f"size_x   = {size[0]:.4f}\n")
    fh.write(f"size_y   = {size[1]:.4f}\n")
    fh.write(f"size_z   = {size[2]:.4f}\n\n")
    fh.write(f"exhaustiveness = {exhaustiveness}\n")
    fh.write(f"num_modes      = {n_poses}\n")
    fh.write(f"seed           = {seed}\n")
    fh.write(f"scoring        = {scoring_fn}\n\n")
    fh.write(f"# {formula}  MW={mw:.2f}  LogP={logp:.2f}  RotB={rotb}\n")


# ---------------------------------------------------------------------------
#  Output: Summary CSV
# ---------------------------------------------------------------------------

print("  Writing validation_results.csv ...")
csv_path = WORKDIR / "validation_results.csv"
_fields = [
    "Pose", "Affinity_kcal_mol", "RMSD_vs_crystal_A",
    "RMSD_lb_A", "RMSD_ub_A", "Status",
    "Ligand_name", "Formula", "MW_Da", "LogP", "HBD", "HBA",
    "RotB", "TPSA_A2", "Lipinski_Ro5",
    "Scoring_fn", "Exhaustiveness", "Box_padding_A",
]
with open(csv_path, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=_fields)
    writer.writeheader()
    for r in rmsd_results:
        writer.writerow({
            "Pose":              r["pose"],
            "Affinity_kcal_mol": round(r["affinity"], 4),
            "RMSD_vs_crystal_A": round(r["rmsd"], 4) if r["rmsd"] is not None else "N/A",
            "RMSD_lb_A":         round(r["rmsd_lb"], 4),
            "RMSD_ub_A":         round(r["rmsd_ub"], 4),
            "Status":            r["status"],
            "Ligand_name":       lig_name,
            "Formula":           formula,
            "MW_Da":             round(mw, 3),
            "LogP":              round(logp, 3),
            "HBD":               hbd,
            "HBA":               hba,
            "RotB":              rotb,
            "TPSA_A2":           round(tpsa, 2),
            "Lipinski_Ro5":      "PASS" if ro5_ok else "FAIL",
            "Scoring_fn":        scoring_fn,
            "Exhaustiveness":    exhaustiveness,
            "Box_padding_A":     pad_ref,
        })
print(f"  [OK]  {len(rmsd_results)} row(s) -> validation_results.csv")


# ---------------------------------------------------------------------------
#  Output: Per-pose PDB files (optional)
# ---------------------------------------------------------------------------

if export_pose_pdbs:
    print("\n  Writing per-pose PDB files ...")
    pd_dir = lig_dir / "individual_poses"
    pd_dir.mkdir(exist_ok=True)
    total_pdbs = 0

    if out_pdbqt.exists():
        # Extract per-pose affinities from PDBQT remarks
        affs = []
        with open(out_pdbqt) as fh:
            for ln in fh:
                if "VINA RESULT" in ln:
                    try:
                        affs.append(float(ln.split()[2]))
                    except (IndexError, ValueError):
                        affs.append(0.0)

        raw = open(out_pdbqt).read()
        for blk in raw.split("MODEL")[1:]:
            lns = blk.strip().splitlines()
            try:
                mid = int(lns[0].strip())
            except (IndexError, ValueError):
                continue
            aff = affs[mid - 1] if mid - 1 < len(affs) else 0.0
            rms_val = rmsd_results[mid - 1]["rmsd"] if mid - 1 < len(rmsd_results) else None
            rms_str = f"RMSD={rms_val:.3f}A" if rms_val is not None else "RMSD=N/A"

            with open(pd_dir / f"pose_{mid:02d}.pdb", "w") as fh:
                fh.write(f"REMARK {lig_name} Pose {mid} {aff:.3f} kcal/mol {rms_str}\n")
                fh.writelines(l + "\n" for l in lns[1:]
                              if l.startswith(("ATOM", "HETATM")))
                fh.write("END\n")
            total_pdbs += 1

    print(f"  [OK]  {total_pdbs} PDB file(s) written")


# ---------------------------------------------------------------------------
#  Output: ZIP archive
# ---------------------------------------------------------------------------

print(f"\n  Packaging {run_name}_results.zip ...")
out_zip = Path(f"/content/{run_name}_results.zip")
zip_top = f"{run_name}_results"

with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in [csv_path, prot_pdbqt]:
        if f and f.exists():
            zf.write(f, f"{zip_top}/{f.name}")
    # Include co-crystal reference
    zf.write(cocrys_path, f"{zip_top}/reference_cocrystal{cocrys_ext}")
    # Include everything in ligand_dock/
    for fp in sorted(lig_dir.rglob("*")):
        if fp.is_file():
            zf.write(fp, f"{zip_top}/{fp.relative_to(WORKDIR)}")

print(f"  [OK]  {out_zip.stat().st_size / 1024:.1f} KB -> {out_zip.name}")


# ---------------------------------------------------------------------------
#  Summary Banner
# ---------------------------------------------------------------------------

best_r  = best_rmsd_row if valid_rmsd else None
verdict = "PASSED" if (best_r and best_r["rmsd"] <= rmsd_threshold) else "FAILED"

print("\n" + "=" * 72)
print("  COGNATE DOCKING VALIDATION COMPLETE")
print("=" * 72)
print(f"  Run name        : {run_name}")
print(f"  Protein         : {prot_orig}")
print(f"  Co-crystal      : {cocrys_orig}")
print(f"  Clean ligand    : {clean_orig}")
print(f"  Scoring fn      : {scoring_fn}   Exhaustiveness : {exhaustiveness}")
print(f"  Poses generated : {len(docked_mols)}   Runtime : {elapsed:.1f}s")
if best_r:
    print(f"  Best RMSD       : {best_r['rmsd']:.3f} A  (Pose {best_r['pose']})")
    print(f"  Best Affinity   : {best_aff_row['affinity']:.3f} kcal/mol  (Pose {best_aff_row['pose']})")
    n_pass = sum(1 for r in valid_rmsd if r["rmsd"] <= rmsd_threshold)
    print(f"  Threshold       : {rmsd_threshold} A   Passes : {n_pass}/{len(valid_rmsd)}")
print(f"  Verdict         : {verdict}")
print("-" * 72)
print(f"  ZIP contents:")
print(f"    validation_results.csv  -- all poses with RMSD and affinity")
print(f"    reference_cocrystal.*   -- crystal pose (ground truth)")
print(f"    protein.pdbqt           -- converted receptor")
print(f"    ligand_dock/            -- docked poses, config, individual PDBs")
print("-" * 72)
print(f"  Visualise: PyMOL, Chimera, PLIP, LigPlot+")
print(f"  Tip: load co-crystal + best pose in PyMOL to compare")
print("=" * 72)


# ---------------------------------------------------------------------------
#  3D Visualisation (py3Dmol)
# ---------------------------------------------------------------------------

print("\n" + "=" * 72)
print("  3D VISUALISATION (py3Dmol)")
print("=" * 72)

_pip("py3Dmol")
import py3Dmol


def _mol_to_molblock(rdmol):
    """Return an SDF mol block string for py3Dmol."""
    return Chem.MolToMolBlock(rdmol)


# Identify best-RMSD and rank-1 (best affinity) poses
best_pose_idx = best_rmsd_row["pose"] - 1 if valid_rmsd else 0
best_docked_mol = docked_mols[best_pose_idx] if best_pose_idx < len(docked_mols) else docked_mols[0]

rank1_idx = 0  # first pose = best affinity from Vina
rank1_mol = docked_mols[rank1_idx]
show_both_poses = (best_pose_idx != rank1_idx) and valid_rmsd

# Read protein and prepare mol blocks
prot_pdb_str         = open(prot_in).read()
ref_molblock         = _mol_to_molblock(ref_mol)
best_docked_molblock = _mol_to_molblock(best_docked_mol)
rank1_molblock       = _mol_to_molblock(rank1_mol)

_br = best_rmsd_row["rmsd"] if valid_rmsd else 0
_bp = best_rmsd_row["pose"]  if valid_rmsd else 1


# -- View 1: Best-RMSD pose vs Crystal ---------------------------------------

print(f"\n  View 1 -- Best-RMSD Pose (Pose {_bp}, RMSD = {_br:.3f} A) vs Crystal\n")

view1 = py3Dmol.view(width=900, height=600)

# Protein: white cartoon + faint surface
view1.addModel(prot_pdb_str, "pdb")
view1.setStyle({"model": 0}, {"cartoon": {"color": "white", "opacity": 0.85}})
view1.addSurface(py3Dmol.VDW,
                 {"opacity": 0.08, "color": "lightgray"},
                 {"model": 0})

# Co-crystallised ligand: green sticks
view1.addModel(ref_molblock, "sdf")
view1.setStyle({"model": 1},
               {"stick": {"colorscheme": "greenCarbon", "radius": 0.20}})

# Best-RMSD docked pose: magenta sticks
view1.addModel(best_docked_molblock, "sdf")
view1.setStyle({"model": 2},
               {"stick": {"colorscheme": "magentaCarbon", "radius": 0.20}})

view1.zoomTo({"model": 1})
view1.addLabel("Crystal (green)",
               {"backgroundColor": "green", "fontColor": "white",
                "fontSize": 12, "showBackground": True,
                "backgroundOpacity": 0.8},
               {"model": 1, "serial": 1})
view1.addLabel(f"Docked Pose {_bp} (magenta)\nRMSD = {_br:.2f} A",
               {"backgroundColor": "#cc00cc", "fontColor": "white",
                "fontSize": 12, "showBackground": True,
                "backgroundOpacity": 0.8},
               {"model": 2, "serial": 1})
view1.show()


# -- View 2: Rank-1 vs Crystal (only if different from best-RMSD) ------------

if show_both_poses:
    _r1_rmsd = rmsd_results[rank1_idx]["rmsd"]
    _r1_aff  = rmsd_results[rank1_idx]["affinity"]
    _r1_rms_str = f"{_r1_rmsd:.3f}" if _r1_rmsd is not None else "N/A"

    print(f"\n  View 2 -- Rank-1 / Best Affinity (Pose 1, "
          f"RMSD = {_r1_rms_str} A, {_r1_aff:.3f} kcal/mol) vs Crystal\n")

    view2 = py3Dmol.view(width=900, height=600)
    view2.addModel(prot_pdb_str, "pdb")
    view2.setStyle({"model": 0}, {"cartoon": {"color": "white", "opacity": 0.85}})
    view2.addSurface(py3Dmol.VDW,
                     {"opacity": 0.08, "color": "lightgray"},
                     {"model": 0})

    view2.addModel(ref_molblock, "sdf")
    view2.setStyle({"model": 1},
                   {"stick": {"colorscheme": "greenCarbon", "radius": 0.20}})

    view2.addModel(rank1_molblock, "sdf")
    view2.setStyle({"model": 2},
                   {"stick": {"colorscheme": "cyanCarbon", "radius": 0.20}})

    view2.zoomTo({"model": 1})
    view2.addLabel("Crystal (green)",
                   {"backgroundColor": "green", "fontColor": "white",
                    "fontSize": 12, "showBackground": True,
                    "backgroundOpacity": 0.8},
                   {"model": 1, "serial": 1})
    view2.addLabel(f"Rank-1 / Pose 1 (cyan)\nRMSD = {_r1_rms_str} A",
                   {"backgroundColor": "#009999", "fontColor": "white",
                    "fontSize": 12, "showBackground": True,
                    "backgroundOpacity": 0.8},
                   {"model": 2, "serial": 1})
    view2.show()


# -- View 3: All three overlaid (only if best-RMSD != rank-1) ----------------

if show_both_poses:
    print(f"\n  View 3 -- All overlaid: Crystal (green), "
          f"Best-RMSD Pose {_bp} (magenta), Rank-1 Pose 1 (cyan)\n")

    view3 = py3Dmol.view(width=900, height=600)
    view3.addModel(prot_pdb_str, "pdb")
    view3.setStyle({"model": 0}, {"cartoon": {"color": "white", "opacity": 0.85}})
    view3.addSurface(py3Dmol.VDW,
                     {"opacity": 0.08, "color": "lightgray"},
                     {"model": 0})

    view3.addModel(ref_molblock, "sdf")
    view3.setStyle({"model": 1},
                   {"stick": {"colorscheme": "greenCarbon", "radius": 0.22}})

    view3.addModel(best_docked_molblock, "sdf")
    view3.setStyle({"model": 2},
                   {"stick": {"colorscheme": "magentaCarbon", "radius": 0.18}})

    view3.addModel(rank1_molblock, "sdf")
    view3.setStyle({"model": 3},
                   {"stick": {"colorscheme": "cyanCarbon", "radius": 0.18}})

    view3.zoomTo({"model": 1})
    view3.addLabel("Crystal",
                   {"backgroundColor": "green", "fontColor": "white",
                    "fontSize": 11, "showBackground": True},
                   {"model": 1, "serial": 1})
    view3.addLabel(f"Best RMSD (P{_bp})",
                   {"backgroundColor": "#cc00cc", "fontColor": "white",
                    "fontSize": 11, "showBackground": True},
                   {"model": 2, "serial": 1})
    view3.addLabel("Rank-1 (P1)",
                   {"backgroundColor": "#009999", "fontColor": "white",
                    "fontSize": 11, "showBackground": True},
                   {"model": 3, "serial": 1})
    view3.show()


# -- View 4: Binding-site zoom -----------------------------------------------

print(f"\n  View 4 -- Binding-site zoom (protein residues within 5 A)\n")

view4 = py3Dmol.view(width=900, height=600)

view4.addModel(prot_pdb_str, "pdb")
view4.setStyle({"model": 0}, {"cartoon": {"color": "white", "opacity": 0.4}})
view4.addStyle({"model": 0,
                "byres": True,
                "within": {"distance": 5.0, "sel": {"model": 1}}},
               {"stick": {"colorscheme": "whiteCarbon", "radius": 0.12},
                "label": {"fontSize": 10, "showBackground": False}})

view4.addModel(ref_molblock, "sdf")
view4.setStyle({"model": 1},
               {"stick": {"colorscheme": "greenCarbon", "radius": 0.22}})

view4.addModel(best_docked_molblock, "sdf")
view4.setStyle({"model": 2},
               {"stick": {"colorscheme": "magentaCarbon", "radius": 0.22}})

view4.zoomTo({"model": 1})
view4.addLabel("Crystal (green)",
               {"backgroundColor": "green", "fontColor": "white",
                "fontSize": 11, "showBackground": True,
                "backgroundOpacity": 0.8},
               {"model": 1, "serial": 1})
view4.addLabel(f"Docked (magenta) RMSD = {_br:.2f} A",
               {"backgroundColor": "#cc00cc", "fontColor": "white",
                "fontSize": 11, "showBackground": True,
                "backgroundOpacity": 0.8},
               {"model": 2, "serial": 1})
view4.show()

print(f"""
  Legend:
    Green sticks   = Co-crystallised pose (reference)
    Magenta sticks = Best-RMSD docked pose
    Cyan sticks    = Rank-1 / best affinity pose
    White cartoon  = Protein backbone

  Controls:
    Left-click drag  = rotate
    Scroll           = zoom
    Right-click drag = translate
""")


# ---------------------------------------------------------------------------
#  Download
# ---------------------------------------------------------------------------

print(f"  Downloading {out_zip.name} ...")
files.download(str(out_zip))
print("  Check your browser's download bar.")


  STEP 1/7  |  Installing Dependencies
  [OK]  vina                   AutoDock Vina 1.2.x Python bindings
  [OK]  openbabel-wheel        Format conversion (PDB/SDF/MOL2 -> PDBQT)
  [OK]  scipy                  Hungarian algorithm for symmetry-corrected RMSD
  [OK]  rdkit                  already available
  All dependencies ready.

  STEP 2/7  |  File Upload
  Protein (.pdb) -- cleaned, ligand/water removed:


Saving 09_apo_protein_clean_FINAL.pdb to 09_apo_protein_clean_FINAL.pdb
  [OK]  09_apo_protein_clean_FINAL.pdb  (167.9 KB)

  Co-crystallised ligand (.sdf / .mol2 / .pdb) -- the crystal pose:


Saving xtal_ligand.sdf to xtal_ligand (3).sdf
  [OK]  xtal_ligand (3).sdf  (2.9 KB)

  Clean ligand (.sdf / .mol2) -- will be redocked:
  (Same molecule, fresh 3D conformer or energy-minimised)


Saving crystallized_ligand_clean.sdf to crystallized_ligand_clean (1).sdf
  [OK]  crystallized_ligand_clean (1).sdf  (4.7 KB)

  STEP 3/7  |  Parsing Co-crystallised Ligand and Defining Box
  Co-crystallised ligand : C23H25ClN4O2S
  Atoms                  : 31 total (31 heavy)
  Centroid               : (28.75, 15.83, -2.34) A
  Box size               : (19.29, 19.84, 20.49) A  [padding = 6.0 A]
  Box volume             : 7,841 A^3

  STEP 4/7  |  Protein Conversion (PDB -> PDBQT)
  [4a] Parsing coordinates ...
       2,121 atoms, 127 residues
  [4b] PDB -> PDBQT (OpenBabel, Gasteiger charges, rigid) ...
       1,295 PDBQT records -> protein.pdbqt

  STEP 5/7  |  Parsing Clean Ligand
  Name         : LIGAND
  Formula      : C23H25ClN4O2S   MW = 456.1 Da
  LogP = 5.53   HBD = 0   HBA = 6   RotB = 3   TPSA = 69 A^2
  Lipinski Ro5 : PASS
  Atoms        : 56 total (31 heavy)
  3D coords    : confirmed

  [OK]  Canonical SMILES match -- same molecule confirmed

  STEP 6/7  |  Settings Confi

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

...

3Dmol.js failed to load for some reason. Please check your browser console for error messages.


  Legend:
    Green sticks   = Co-crystallised pose (reference)
    Magenta sticks = Best-RMSD docked pose
    Cyan sticks    = Rank-1 / best affinity pose
    White cartoon  = Protein backbone

  Controls:
    Left-click drag  = rotate
    Scroll           = zoom
    Right-click drag = translate



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Check your browser's download bar.
